Regional Input-Output Table Calculator

**Note to Users:** Before running this notebook, please ensure you have installed the required packages listed in the `requirements.txt` file.

- If you are using a Jupyter Notebook environment running python, you should run the following command on your console:

pip install -r requirements.txt

- If you are using an rmarkdown environment, run in your console the following command:

reticulate::py_install(c("numpy", "scipy", "pandas", "matplotlib", "openpyxl", "pyxlsb", "pandasgui"))

- It may take a few minutes

In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pyxlsb
import re
import os

An example with a small IO table:

In [2]:
# As soon as we have the cleaned dataset, we will be able to perform calculations.
# 0)
# First we obtain our IO matrix, the added value matrix and the final demand matrix

IO = np.array([[2, 1, 3],
              [3, 2, 1],
              [1, 1, 1]])

VA = np.array([[3, 1, 1],
               [1, 3, 0]])

Y = np.array([[3, 1],
              [1 , 1],
              [2, 1]])

# 1)
# Then, we compute the gross production value vector (Valor Bruto de Produção - VBP). We basically sum the elements in the IO matrix by lines, and sum the elements in the
# added value matrix also by lines. As a result, we obtain the transposed vector. 

VBP_T = np.sum(IO, axis=0) + np.sum(VA, axis=0)


# 2)
# Now, we need to compute the coeficient matrix. For that, we need to divided the elements in the first column of the IO matrix by its respective gross production value,
# which corresponds to the first element of the transposed gross production value vector.


A = IO/VBP_T


# When we perform a division of a matrix by a line vector, python already repeats the division for each line.
# It performs the division element by element (i.e. the nth element in a given line of the matrix is divided by the nth element in the line of the vector)

# 3)
# Then, we create the identity matrix with the same dimension 

I = np.eye(3)


# 4)
# We subtract the identity matrix by the coefficient matrix and invert it.

L = np.linalg.inv(I - A)

# And we are done. 

# 5)
# We can also multiply the Leontieff inverse matrix by the final demand vector to generate the gross production value vector to check if the calculations
# were performed correctly:

error_vector = VBP_T - L @ np.sum(Y, axis = 1)

# 6)
# We can also sum the absolute value of the coordinates of the error vector to obtain a measure of error in the data. They are usually residual and very small, which is the
# case here:

absolute_sum = np.sum(abs(error_vector))

#############################################################################################################################################################################

# This is just the code to print the result the values of each step:

print('1) Gross production value vector: ')
print(VBP_T)
print('')
print('2) Coefficient matrix: ')
print(A)
print('')
print('3) Identity matrix: ')
print(I)
print('')
print('4) Leontieff inverse matrix')
print(L)
print('')
print('5) Error vector:')
print(error_vector)
print('')
print('6) Absolute sum of the errors:')
print(absolute_sum)

1) Gross production value vector: 
[10  8  6]

2) Coefficient matrix: 
[[0.2        0.125      0.5       ]
 [0.3        0.25       0.16666667]
 [0.1        0.125      0.16666667]]

3) Identity matrix: 
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

4) Leontieff inverse matrix
[[1.53439153 0.42328042 1.00529101]
 [0.67724868 1.56613757 0.71957672]
 [0.28571429 0.28571429 1.42857143]]

5) Error vector:
[0.0000000e+00 0.0000000e+00 8.8817842e-16]

6) Absolute sum of the errors:
8.881784197001252e-16


PART 0:

Now we will import our IO-table to the Jupyter environment:

Importing the dataframe:

In [2]:
print(os.listdir())

# =======================================================
# USER INPUT REQUIRED:
# Change the year below to the year you want to analyze.
# =======================================================

file_year = "2009"  

# The code will automatically build the file name

file_name = f"RegionalIOtable_{file_year}.xlsb"

file_relative_path = f"data/raw/{file_name}"

print(f"Loading data from {file_name}...")

df = pd.read_excel(
    file_relative_path,
    engine="pyxlsb",
    sheet_name=file_year,
    header=None,
)
print("Data loaded successfully!")
df

['.git', '.gitattributes', '.gitignore', 'code_v1.ipynb', 'code_v1.Rmd', 'convert_to_rmarkdown', 'data', 'legacy_code.ipynb', 'README.md', 'requirements.txt']
Loading data from RegionalIOtable_2009.xlsb...
Data loaded successfully!


,0,1,2,3,4,5,6,7,8,9,...,4791,4792,4793,4794,4795,4796,4797,4798,4799,4800
0,Return to Index,NaN,NaN,Regional IOtable of the year 2009,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Region Code A,Region Code B,Region Name,IO row Code,IO row name,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,R-Code A,NaN,NaN,NaN,NaN,NaN,R1,R1,R1,R1,...,L46,L46,L47,L47,L47,L47,L48,L48,L48,L48
4,R-Code B,NaN,NaN,NaN,NaN,NaN,AT11,AT11,AT11,AT11,...,TUR,TUR,USA,USA,USA,USA,TWN,TWN,TWN,TWN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3738,NaN,NaN,NaN,NaN,IO row Code,IO row name,Agriculture,Mining_quarrying_and_energy_supply,Food_beverages_and_tobacco,Textiles_and_leather_etc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3739,NaN,NaN,NaN,NaN,sp60,Compensation_of_employees,16.82145,59.619464,58.876294,13.744242,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3740,NaN,NaN,NaN,NaN,sp61,Other_value_added,148.117717,153.736256,83.816406,10.747893,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3741,NaN,NaN,NaN,NaN,sp62,Taxes_less_subsidies_on_products,12.480693,34.996912,17.728262,4.008108,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Name of the file: 'RegionalIOtable_2009.xlsb'

Name of the sheet: '2009'

We separate the bigger dataframe into smaller ones so we can work with each of them individually:


In [3]:
# # The IO table data starts on row 4 and ends on row 3732.
# Column 0 is the index, so we start at column 1. 
# It ends in column 3730

START_ROW = 3 
END_ROW = 3732
START_COL = 1
END_COL = 3730

tabela_IO = df.iloc[START_ROW:END_ROW, START_COL:END_COL]
tabela_IO

,1,2,3,4,5,6,7,8,9,10,...,3720,3721,3722,3723,3724,3725,3726,3727,3728,3729
3,NaN,NaN,NaN,NaN,NaN,R1,R1,R1,R1,R1,...,L48,L48,L48,L48,L48,L48,L48,L48,L48,L48
4,NaN,NaN,NaN,NaN,NaN,AT11,AT11,AT11,AT11,AT11,...,TWN,TWN,TWN,TWN,TWN,TWN,TWN,TWN,TWN,TWN
5,NaN,NaN,NaN,NaN,NaN,Burgenland,Burgenland,Burgenland,Burgenland,Burgenland,...,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan
6,NaN,NaN,NaN,NaN,NaN,ss1,ss2,ss3,ss4,ss5,...,ss5,ss6,ss8,ss9,ss10,ss11,ss12,ss13,ss14,ss15
7,NaN,NaN,NaN,NaN,NaN,Agriculture,Mining_quarrying_and_energy_supply,Food_beverages_and_tobacco,Textiles_and_leather_etc,Coke_refined_petroleum_nuclear_fuel_and_chemic...,...,Coke_refined_petroleum_nuclear_fuel_and_chemic...,Electrical_and_optical_equipment_and_Transport...,Other_manufacturing,Construction,Distribution,Hotels_and_restaurant,Transport_storage_and_communication,Financial_intermediation,Real_estate_renting_and_busine_activitie,Non-Market_Service
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3727,L48,TWN,Taiwan,ss11,Hotels_and_restaurant,0.001063,0.003447,0.006365,0.000615,0.002894,...,111.320075,97.307843,129.929373,30.494573,172.411134,2.664454,108.325738,92.566268,152.705406,515.037023
3728,L48,TWN,Taiwan,ss12,Transport_storage_and_communication,0.002813,0.021459,0.021536,0.002512,0.013895,...,451.845994,1249.268469,1125.812455,464.840264,800.267228,64.757076,3579.524786,308.397311,731.732177,1710.922433
3729,L48,TWN,Taiwan,ss13,Financial_intermediation,0.002276,0.002313,0.002037,0.000229,0.001777,...,683.267129,2234.974885,1345.340408,328.545598,844.461909,74.811342,376.903829,3375.227689,1305.035553,4846.28085
3730,L48,TWN,Taiwan,ss14,Real_estate_renting_and_busine_activitie,0.008728,0.022872,0.043662,0.004078,0.021797,...,708.221958,3699.497133,1398.515545,904.863485,12388.334313,627.209415,1638.753778,2553.209722,4223.006661,4192.84656


In [4]:
# # The Y table data starts on row 4 and ends on row 3732.
# Column 3732 is the end of the IO table, and python starts at the following term.
# It ends on column 4801.

START_ROW = 3 
END_ROW = 3732
START_COL = 3732
END_COL = 4801


tabela_Y = df.iloc[START_ROW:END_ROW, START_COL:END_COL]
tabela_Y

,3732,3733,3734,3735,3736,3737,3738,3739,3740,3741,...,4791,4792,4793,4794,4795,4796,4797,4798,4799,4800
3,NaN,NaN,NaN,NaN,NaN,R1,R1,R1,R1,R2,...,L46,L46,L47,L47,L47,L47,L48,L48,L48,L48
4,NaN,NaN,NaN,NaN,NaN,AT11,AT11,AT11,AT11,AT12,...,TUR,TUR,USA,USA,USA,USA,TWN,TWN,TWN,TWN
5,NaN,NaN,NaN,NaN,NaN,Burgenland,Burgenland,Burgenland,Burgenland,Niederosterreich,...,Turkey,Turkey,United_States,United_States,United_States,United_States,Taiwan,Taiwan,Taiwan,Taiwan
6,NaN,NaN,NaN,NaN,NaN,ss16,ss17,ss18,ss19,ss16,...,ss18,ss19,ss16,ss17,ss18,ss19,ss16,ss17,ss18,ss19
7,NaN,NaN,NaN,NaN,NaN,Final_consumption_expenditure_by_households_an...,Final_consumption_expenditure_by_government,Net_capital_formation,Inventory_adjustment,Final_consumption_expenditure_by_households_an...,...,Net_capital_formation,Inventory_adjustment,Final_consumption_expenditure_by_households_an...,Final_consumption_expenditure_by_government,Net_capital_formation,Inventory_adjustment,Final_consumption_expenditure_by_households_an...,Final_consumption_expenditure_by_government,Net_capital_formation,Inventory_adjustment
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3727,L48,TWN,Taiwan,ss11,Hotels_and_restaurant,0.122558,0.00045,NaN,NaN,0.703073,...,NaN,NaN,NaN,NaN,NaN,NaN,5393.189984,20.03807,NaN,0.024985
3728,L48,TWN,Taiwan,ss12,Transport_storage_and_communication,0.190314,0.00382,0.000893,NaN,1.107727,...,NaN,NaN,292.872736,NaN,15.900591,NaN,11892.172053,53.52201,253.501291,-22.462273
3729,L48,TWN,Taiwan,ss13,Financial_intermediation,0.03057,NaN,NaN,NaN,0.17219,...,NaN,NaN,57.568826,NaN,NaN,NaN,14214.320182,198.300674,NaN,0.011723
3730,L48,TWN,Taiwan,ss14,Real_estate_renting_and_busine_activitie,0.095238,0.011663,NaN,NaN,0.554061,...,NaN,NaN,73.247635,NaN,NaN,NaN,780.408698,438.896587,8314.920236,-0.576978


In [5]:
# # The VA table data starts after the IO table, on row 3735 (so we put 3734) and ends on row 3744.
# Column 0 is the index, so we start at column 1. 
# It ends in column 3730

START_ROW = 3734 
END_ROW = 3744
START_COL = 1
END_COL = 3730

tabela_VA = df.iloc[START_ROW:END_ROW, START_COL:END_COL]
tabela_VA

,1,2,3,4,5,6,7,8,9,10,...,3720,3721,3722,3723,3724,3725,3726,3727,3728,3729
3734,NaN,NaN,NaN,NaN,NaN,R1,R1,R1,R1,R1,...,L48,L48,L48,L48,L48,L48,L48,L48,L48,L48
3735,NaN,NaN,NaN,NaN,NaN,AT11,AT11,AT11,AT11,AT11,...,TWN,TWN,TWN,TWN,TWN,TWN,TWN,TWN,TWN,TWN
3736,NaN,NaN,NaN,NaN,NaN,Burgenland,Burgenland,Burgenland,Burgenland,Burgenland,...,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan,Taiwan
3737,NaN,NaN,NaN,NaN,NaN,ss1,ss2,ss3,ss4,ss5,...,ss5,ss6,ss8,ss9,ss10,ss11,ss12,ss13,ss14,ss15
3738,NaN,NaN,NaN,IO row Code,IO row name,Agriculture,Mining_quarrying_and_energy_supply,Food_beverages_and_tobacco,Textiles_and_leather_etc,Coke_refined_petroleum_nuclear_fuel_and_chemic...,...,Coke_refined_petroleum_nuclear_fuel_and_chemic...,Electrical_and_optical_equipment_and_Transport...,Other_manufacturing,Construction,Distribution,Hotels_and_restaurant,Transport_storage_and_communication,Financial_intermediation,Real_estate_renting_and_busine_activitie,Non-Market_Service
3739,NaN,NaN,NaN,sp60,Compensation_of_employees,16.82145,59.619464,58.876294,13.744242,54.430339,...,6850.993281,12107.182636,8004.436598,2939.460114,24980.362202,3291.124615,5715.55421,10687.405576,8528.48278,40887.884792
3740,NaN,NaN,NaN,sp61,Other_value_added,148.117717,153.736256,83.816406,10.747893,81.767108,...,9184.245807,8626.368045,6280.780458,2774.288739,25105.294337,2623.057999,8676.531996,12664.666714,23097.200558,18205.268262
3741,NaN,NaN,NaN,sp62,Taxes_less_subsidies_on_products,12.480693,34.996912,17.728262,4.008108,15.452907,...,542.417372,816.497885,430.384184,189.231488,192.174541,36.765966,168.581187,29.931123,134.020426,291.573545
3742,NaN,NaN,NaN,sp65,International_trade_margin,2.475388,10.405807,2.579398,1.521987,21.782832,...,1197.380911,1618.266842,669.983535,200.402601,65.149956,13.446121,89.212078,4.652733,66.233716,149.433636


Notice that the labels of the files are considered as part of the data. We need to fix this, using a multindex for both rows and columns. Creating the multindex for each dataframe

In [6]:
# In the IO table, the first 5 columns and first 5 rows give us two regional codes, a region name, a sector name and a sector name

rscn = ['region1', 'region2', 'region3', 'sector1','sector2'] # regional-sectors' code names

START_ROW = 0 
END_ROW = 5
START_COL = 0
END_COL = 5

# The rows' index start after the 5th row and go to the bottom of the DF. They are the first 5 columns of the DF 

row_IO = pd.MultiIndex.from_frame(
    tabela_IO.iloc[END_ROW:, START_COL:END_COL], names = rscn
)

# The columns' index start after the 5th column and go to the extreme right of the DF. They are the first rows of the DF

col_IO = pd.MultiIndex.from_arrays(
    tabela_IO.iloc[START_ROW:END_ROW, END_COL:].values, names = rscn
)

# The data starts after the 5h row and after the 5th column:

IO_values = tabela_IO.iloc[END_ROW:, END_COL:].astype(float)

df_IO = pd.DataFrame(
    IO_values.values,
    index=row_IO,
    columns=col_IO
)

df_IO

region1                                                                                        R1  \
region2                                                                                      AT11   
region3                                                                                Burgenland   
sector1                                                                                       ss1   
sector2                                                                               Agriculture   
region1 region2 region3    sector1 sector2                                                          
R1      AT11    Burgenland ss1     Agriculture                                          62.431109   
                           ss2     Mining_quarrying_and_energy_supply                    4.349757   
                           ss3     Food_beverages_and_tobacco                            8.621868   
                           ss4     Textiles_and_leather_etc                              0.099653   
                           ss5     Coke_refined_petroleum_nuclear_fuel_and_chemica...    1.675379   
...                                                                                           ...   
L48     TWN     Taiwan     ss11    Hotels_and_restaurant                                 0.001063   
                           ss12    Transport_storage_and_communication                   0.002813   
                           ss13    Financial_intermediation                              0.002276   
                           ss14    Real_estate_renting_and_busine_activitie              0.008728   
                           ss15    Non-Market_Service                                    0.001638   

region1                                                                                                                   \
region2                                                                                                                    
region3                                                                                                                    
sector1                                                                                                              ss2   
sector2                                                                               Mining_quarrying_and_energy_supply   
region1 region2 region3    sector1 sector2                                                                                 
R1      AT11    Burgenland ss1     Agriculture                                                                  0.079606   
                           ss2     Mining_quarrying_and_energy_supply                                         311.952886   
                           ss3     Food_beverages_and_tobacco                                                   0.292876   
                           ss4     Textiles_and_leather_etc                                                     0.043132   
                           ss5     Coke_refined_petroleum_nuclear_fuel_and_chemica...                           1.282154   
...                                                                                                                  ...   
L48     TWN     Taiwan     ss11    Hotels_and_restaurant                                                        0.003447   
                           ss12    Transport_storage_and_communication                                          0.021459   
                           ss13    Financial_intermediation                                                     0.002313   
                           ss14    Real_estate_renting_and_busine_activitie                                     0.022872   
                           ss15    Non-Market_Service                                                                NaN   

region1                                                                                                           \
region2                                                  

In [7]:
# In the Y table, the first 5 columns and first 5 rows give us two regional codes, a region name, a sector name and a sector name

# regional-sectors' code names: rscn = ['region1', 'region2', 'region3', 'sector1','sector2']  

START_ROW = 0 
END_ROW = 5
START_COL = 0
END_COL = 5

# The rows' index start after the 5th row and go to the bottom of the DF. They are the first 5 columns of the DF 

row_Y = pd.MultiIndex.from_frame(
    tabela_Y.iloc[END_ROW:, START_COL:END_COL], names = rscn
)

# The columns' index start after the 5th column and go to the extreme right of the DF. They are the first rows of the DF

col_Y = pd.MultiIndex.from_arrays(
    tabela_Y.iloc[START_ROW:END_ROW, END_COL:].values, names = rscn
)

# The data starts after the 5h row and after the 5th column:

Y_values = tabela_Y.iloc[END_ROW:, END_COL:].astype(float)

df_Y = pd.DataFrame(
    Y_values.values,
    index=row_Y,
    columns=col_Y
)

df_Y

region1                                                                                                                                           R1  \
region2                                                                                                                                         AT11   
region3                                                                                                                                   Burgenland   
sector1                                                                                                                                         ss16   
sector2                                                                               Final_consumption_expenditure_by_households_and_non-profit_org   
region1 region2 region3    sector1 sector2                                                                                                             
R1      AT11    Burgenland ss1     Agriculture                                                                                 54.165638               
                           ss2     Mining_quarrying_and_energy_supply                                                         104.711078               
                           ss3     Food_beverages_and_tobacco                                                                 119.973601               
                           ss4     Textiles_and_leather_etc                                                                     1.859372               
                           ss5     Coke_refined_petroleum_nuclear_fuel_and_chemica...                                          21.029095               
...                                                                                                                                              ...   
L48     TWN     Taiwan     ss11    Hotels_and_restaurant                                                                        0.122558               
                           ss12    Transport_storage_and_communication                                                          0.190314               
                           ss13    Financial_intermediation                                                                     0.030570               
                           ss14    Real_estate_renting_and_busine_activitie                                                     0.095238               
                           ss15    Non-Market_Service                                                                           0.953182               

region1                                                                                                                            \
region2                                                                                                                             
region3                                                                                                                             
sector1                                                                                                                      ss17   
sector2                                                                               Final_consumption_expenditure_by_government   
region1 region2 region3    sector1 sector2                                                                                          
R1      AT11    Burgenland ss1     Agriculture                                                                           1.389895   
                           ss2     Mining_quarrying_and_energy_supply                                                    0.659726   
                           ss3     Food_beverages_and_tobacco                                                            0.538950   
                           ss4     Textiles_and_leather_etc                                                              0.095684   
                           ss5     Coke_refined_petroleum_nuclear_fuel_and_chemica...

In [8]:
# In the VA table, the first three columns are empty, while the fourth gives us a sector code, and the fifth gives us a sector name
# The first 5 rows give us two regional codes, a region name, a sector name and a sector name


# regional-sectors' code names: rscn = ['region1', 'region2', 'region3', 'sector1','sector2']
rscn1 = ['sector1','sector2'] # sectors' code names

START_ROW = 0 
END_ROW = 5
START_COL = 0
START_COL1 = 3
END_COL = 5

# The rows' index start after the 5th row and go to the bottom of the DF. They are the columns 4 and 5 of the DF 


row_VA = pd.MultiIndex.from_frame(
    tabela_VA.iloc[END_ROW:, START_COL1:END_COL], names = rscn1
)

# The columns' index start after the 5th column and go to the extreme right of the DF. They are the first rows of the DF

col_VA = pd.MultiIndex.from_arrays(
    tabela_VA.iloc[START_ROW:END_ROW, END_COL:].values, names = rscn
)

# The data starts after the 5h row and after the 5th column:

VA_values = tabela_VA.iloc[END_ROW:, END_COL:].astype(float)

df_VA = pd.DataFrame(
    VA_values.values,
    index=row_VA,
    columns=col_VA
)

df_VA

region1                                           R1  \
region2                                         AT11   
region3                                   Burgenland   
sector1                                          ss1   
sector2                                  Agriculture   
sector1 sector2                                        
sp60    Compensation_of_employees          16.821450   
sp61    Other_value_added                 148.117717   
sp62    Taxes_less_subsidies_on_products   12.480693   
sp65    International_trade_margin          2.475388   

region1                                                                      \
region2                                                                       
region3                                                                       
sector1                                                                 ss2   
sector2                                  Mining_quarrying_and_energy_supply   
sector1 sector2                                                               
sp60    Compensation_of_employees                                 59.619464   
sp61    Other_value_added                                        153.736256   
sp62    Taxes_less_subsidies_on_products                          34.996912   
sp65    International_trade_margin                                10.405807   

region1                                                              \
region2                                                               
region3                                                               
sector1                                                         ss3   
sector2                                  Food_beverages_and_tobacco   
sector1 sector2                                                       
sp60    Compensation_of_employees                         58.876294   
sp61    Other_value_added                                 83.816406   
sp62    Taxes_less_subsidies_on_products                  17.728262   
sp65    International_trade_margin                         2.579398   

region1                                                            \
region2                                                             
region3                                                             
sector1                                                       ss4   
sector2                                  Textiles_and_leather_etc   
sector1 sector2                                                     
sp60    Compensation_of_employees                       13.744242   
sp61    Other_value_added                               10.747893   
sp62    Taxes_less_subsidies_on_products                 4.008108   
sp65    International_trade_margin                       1.521987   

region1                                                                                         \
region2                                                                                          
region3                                                                                          
sector1                                                                                    ss5   
sector2                                  Coke_refined_petroleum_nuclear_fuel_and_chemicals_etc   
sector1 sector2                                                                                  
sp60    Compensation_of_employees                                                 54.430339      
sp61    Other_value_added                                                         81.767108      
sp62    Taxes_less_subsidies_on_products                                          15.452907      
sp65    International_trade_margin                                                21.782832      

region1                                                                                            \
region2                                                                                             
region3                                                        

Now that we have a dataframe with the correct indexes, we can perform the mathematical operations described at the beginning.

PART 1: LEGACY CODE, FOR REFERENCE ONLY!

Part 1 is an older approach to the problem, in which we deal with the whole input-output table, in its complete disaggregation. Following parts will deal with an aggregated table for all countries except Italy.

Therefore, we will be dealing with the complete IO matrix, without grouping the intermediary consumption by country and sector. This will give us a bigger Leontieff matrix with much more information. It may be used in the future, but the official data that we shall use is the grouped IO table.

Transforming data into Numpy data type, to perform calculations:

In [36]:
IO = df_IO.to_numpy()
Y = df_Y.to_numpy() 
VA = df_VA.to_numpy()

IO = np.nan_to_num(IO)
Y = np.nan_to_num(Y)
VA = np.nan_to_num(VA)

Technical coefficients matrix:

In [37]:
VBP_T = np.sum(IO, axis=0) + np.sum(VA, axis=0)

A = IO/VBP_T

Identity matrix and Leontieff's inverse matrix:

In [38]:
I = np.eye(A.shape[0])
L = np.linalg.inv(I - A)


Checking values:

In [39]:
Y_s = np.sum(Y, axis = 1) 
Y_s

array([   99.44313841,   185.47516474,   332.52990177, ...,
       14561.01736343,  9849.65671005, 80736.82073548], shape=(3724,))

In [40]:
# Error vector:
error = VBP_T - L @ Y_s

# Sum of the errors:
sum_error = np.sum(error)

# Absolute sum of the errors: 
sum_error_abs = np.sum(abs(error))

# Print values:
print(sum_error, sum_error_abs)

9.381178944778412e-09 9.936782241481978e-08


Creating a dataframe for the technical coefficients matrix and for the leontieff inverse matrix:

In [41]:
df_A = pd.DataFrame(
    A,
    index=row_IO,
    columns=col_IO
)

df_L = pd.DataFrame(
    L,
    index=row_IO,
    columns=col_IO
)

EXPORT DATA FOR PART 1:

In this section you can export the data for each of the tables cleaned and created as a .csv file. Pay attention to the settings used, which is separator is used and decimal, or if either a decimal point or decimal comma is being used. The ideal setting will depend on how your microsoft excel is configured:

Export IO table:

In [42]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "IO.csv"

path = f"data/processed/{name}"

df_IO.to_csv(path , sep = ';', decimal = ',', header = "True", index = "True")

Export final demand table:

In [43]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "Y.csv"

path = f"data/processed/{name}"

df_Y.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export added value table:

In [44]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "VA.csv"

path = f"data/processed/{name}"

df_VA.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export technical coefficients table:

In [45]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "A.csv"

path = f"data/processed/{name}"

df_A.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export the Leontieff's table

In [46]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "L.csv"

path = f"data/processed/{name}"

df_L.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

PART 2:

For this section, we will be reducing the size of the input-output table, final demmand table and added value table. In our tourism dataset, we don't have information from which regions outside of europe each individual is coming from, only from which country. However, we do have the information of how much each tourist spends on each type of expenditure (e.g. housing, restaurants, health services, museums etc.) and in which region of Italy he is spending.

The solution found to this problem was to reduce the whole IO table. For all countries except Italy, we will aggregate all regional-sectors in the table to a single country-level sector. 

For example, if we have a country with just two regions, A and B, we will know from the IO-table that the A-manufacturing industry demands x from the Sicilian-agriculture industry, and the information that the B-manufacturing industry demands y from the same regional-sector. Therefore, in the aggregated table we will have the information that the manufacturing sector from this country demands x+y from the sicilian-agriculture industry. This operation corresponds to summing all columns of the same country and sector. Therefore we perform this operation for all countries except italy.

However, notice that after this first step the information of how much Sicily demands from region A is still mainteined, because we only lost information in the columns. Notice also that we will not lose the information of how much each regional-sector inside Italy provides to each regional-sector inside of Italy, because we are not summing rows and columns for this country.

Likewise, by summing all summing all rows of the same country and sector, we lose the information of how much each regional-sector outside Italy supplies to the Sicilian-agriculture industry. Now, we will only have the information of how much each country supplies to the Sicilian-agriculture sector.

Therefore, our IO table will have four regions: 
1) Countries except Italy X Countries except Italy: How much each sector from each country supplies to each sector from each country;
2) Countries except Italy X Regions of Italy: How much each sector from each country supplies to each sector from each region of Italy;
3) Regions of Italy X Countries except Italy: How much each sector from each region of Italy supplies to each sector from each country;
4) Regions of Italy X Regions of Italy: How much each sector from each region of Italy supplies to each region of Italy.

Performing this operation to the final demand matrix is analogous. Each component of final demand (consumption, government purchases, capital formation and stock formation) are considered to be different sectors. As we have final demand components for each region in Europe, the operation is exactly the same.

For the added-value matrix, while the columns indicate different regional-sectors, the rows indicate payments to different factors of production (capitalists, workers and government) that composed the final market price. The dataset doesn't specify from which country each factor is (e.g. from which country the workers are from), but we can make the simplifying assumption that they are from the same region as the regional-sector. For this matrix, we should only aggregate the columns, as the data in the rows is already aggregated.

First, we make a copy of the original dataframes:

In [9]:
df_IO1 = df_IO.fillna(0)
df_VA1 = df_VA.fillna(0)
df_Y1 = df_Y.fillna(0)

In this step we created a function for transforming the sector code from the format "ss1" to just "1":

In [10]:
# Function that transforms the alphanumerical code in numerical in order not to break the ordering of the final dataframe
def function(data):
    row = []
    sector_order = ['ss1', 'ss2','ss3','ss4','ss5','ss6','ss7','ss8','ss9','ss10','ss11','ss12','ss13','ss14','ss15','ss16','ss17','ss18','ss19']
    for i in data:
        if i in sector_order:
            row.append(sector_order.index(i)+1)
        elif i == 'sp60':
            row.append(60)
        elif i == 'sp61':
            row.append(61)
        elif i == 'sp62':
            row.append(62)
        elif i == 'sp65':
            row.append(65)
        else:
            row.append(99)
    return row

Now we need to add an index for each country. This will make the grouping process easier:

In [11]:
# Separating the regional code:

row_reg_IO1 = df_IO1.index.get_level_values('region2')
col_reg_IO1 = df_IO1.columns.get_level_values('region2')

# The first two characters in the string represent the country code

COUNTRY_CODE_END = 2

row_count_IO1 = row_reg_IO1.str[:COUNTRY_CODE_END]
col_count_IO1 = col_reg_IO1.str[:COUNTRY_CODE_END]

# Applying the function in the sector code

row_sector_IO1 = function(df_IO1.index.get_level_values('sector1'))
col_sector_IO1 = function(df_IO1.columns.get_level_values('sector1'))

rscn2 = ['country','region1', 'region2', 'region3', 'sector1','sector2'] # regional-sector's code name with country

# Creating a multindex with the country code:

row_IO1 = pd.MultiIndex.from_arrays(
    [
        row_count_IO1,
        df_IO1.index.get_level_values('region1'),
        df_IO1.index.get_level_values('region2'),
        df_IO1.index.get_level_values('region3'),
        row_sector_IO1,
        df_IO1.index.get_level_values('sector2'),
    ],
    names = rscn2
)

# Creating a multindex for the columns with the country code 

col_IO1 = pd.MultiIndex.from_arrays(
    [
        col_count_IO1,
        df_IO1.columns.get_level_values('region1'),
        df_IO1.columns.get_level_values('region2'),
        df_IO1.columns.get_level_values('region3'),
        col_sector_IO1,
        df_IO1.columns.get_level_values('sector2'),
    ],
    names = rscn2
)

df_IO2 = pd.DataFrame(
    df_IO1.values,
    index=row_IO1,
    columns=col_IO1
)

df_IO2

country                                                                                                AT  \
region1                                                                                                R1   
region2                                                                                              AT11   
region3                                                                                        Burgenland   
sector1                                                                                                1    
sector2                                                                                       Agriculture   
country region1 region2 region3    sector1 sector2                                                          
AT      R1      AT11    Burgenland 1       Agriculture                                          62.431109   
                                   2       Mining_quarrying_and_energy_supply                    4.349757   
                                   3       Food_beverages_and_tobacco                            8.621868   
                                   4       Textiles_and_leather_etc                              0.099653   
                                   5       Coke_refined_petroleum_nuclear_fuel_and_chemica...    1.675379   
...                                                                                                   ...   
TW      L48     TWN     Taiwan     11      Hotels_and_restaurant                                 0.001063   
                                   12      Transport_storage_and_communication                   0.002813   
                                   13      Financial_intermediation                              0.002276   
                                   14      Real_estate_renting_and_busine_activitie              0.008728   
                                   15      Non-Market_Service                                    0.001638   

country                                                                                                                           \
region1                                                                                                                            
region2                                                                                                                            
region3                                                                                                                            
sector1                                                                                                                       2    
sector2                                                                                       Mining_quarrying_and_energy_supply   
country region1 region2 region3    sector1 sector2                                                                                 
AT      R1      AT11    Burgenland 1       Agriculture                                                                  0.079606   
                                   2       Mining_quarrying_and_energy_supply                                         311.952886   
                                   3       Food_beverages_and_tobacco                                                   0.292876   
                                   4       Textiles_and_leather_etc                                                     0.043132   
                                   5       Coke_refined_petroleum_nuclear_fuel_and_chemica...                           1.282154   
...                                                                                                                          ...   
TW      L48     TWN     Taiwan     11      Hotels_and_restaurant                                                        0.003447   
                                   12      Transport_storage_and_communication                                          0.021459   
                                   13      Financial_inte

In [12]:
row_reg_Y1 = df_Y1.index.get_level_values('region2')
col_reg_Y1 = df_Y1.columns.get_level_values('region2')

COUNTRY_CODE_END = 2

row_count_Y1 = row_reg_Y1.str[:COUNTRY_CODE_END]
col_count_Y1 = col_reg_Y1.str[:COUNTRY_CODE_END]

row_sector_Y1 = function(df_Y1.index.get_level_values('sector1'))
col_sector_Y1 = function(df_Y1.columns.get_level_values('sector1'))

# regional-sector's code name with country: rscn2 = ['country','region1', 'region2', 'region3', 'sector1','sector2'] 

row_Y1 = pd.MultiIndex.from_arrays(
    [
        row_count_Y1,
        df_Y1.index.get_level_values('region1'),
        df_Y1.index.get_level_values('region2'),
        df_Y1.index.get_level_values('region3'),
        row_sector_Y1,
        df_Y1.index.get_level_values('sector2'),
    ],
    names = rscn2
)


col_Y1 = pd.MultiIndex.from_arrays(
    [
        col_count_Y1,
        df_Y1.columns.get_level_values('region1'),
        df_Y1.columns.get_level_values('region2'),
        df_Y1.columns.get_level_values('region3'),
        col_sector_Y1,
        df_Y1.columns.get_level_values('sector2'),
    ],
    names = rscn2
)

df_Y2 = pd.DataFrame(
    df_Y1.values,
    index=row_Y1,
    columns=col_Y1)

df_Y2

country                                                                                                                                                   AT  \
region1                                                                                                                                                   R1   
region2                                                                                                                                                 AT11   
region3                                                                                                                                           Burgenland   
sector1                                                                                                                                                   16   
sector2                                                                                       Final_consumption_expenditure_by_households_and_non-profit_org   
country region1 region2 region3    sector1 sector2                                                                                                             
AT      R1      AT11    Burgenland 1       Agriculture                                                                                 54.165638               
                                   2       Mining_quarrying_and_energy_supply                                                         104.711078               
                                   3       Food_beverages_and_tobacco                                                                 119.973601               
                                   4       Textiles_and_leather_etc                                                                     1.859372               
                                   5       Coke_refined_petroleum_nuclear_fuel_and_chemica...                                          21.029095               
...                                                                                                                                                      ...   
TW      L48     TWN     Taiwan     11      Hotels_and_restaurant                                                                        0.122558               
                                   12      Transport_storage_and_communication                                                          0.190314               
                                   13      Financial_intermediation                                                                     0.030570               
                                   14      Real_estate_renting_and_busine_activitie                                                     0.095238               
                                   15      Non-Market_Service                                                                           0.953182               

country                                                                                                                                    \
region1                                                                                                                                     
region2                                                                                                                                     
region3                                                                                                                                     
sector1                                                                                                                                17   
sector2                                                                                       Final_consumption_expenditure_by_government   
country region1 region2 region3    sector1 sector2                                                                                          
AT      R1      AT11    Burgenland 1       Agriculture                                                                           1.3

In [13]:
col_reg_VA1 = df_VA1.columns.get_level_values('region2')

col_count_VA1 = col_reg_VA1.str[:2]

row_sector_VA1 = function(df_VA1.index.get_level_values('sector1'))
col_sector_VA1 = function(df_VA1.columns.get_level_values('sector1'))

# sectors' code names: rscn1 = ['sector1','sector2'] 

row_VA1 = pd.MultiIndex.from_arrays(
    [
        row_sector_VA1,
        df_VA1.index.get_level_values('sector2')
    ],
    names = rscn1
)

# regional-sector's code name with country: rscn2 = ['country','region1', 'region2', 'region3', 'sector1','sector2'] 


col_VA1 = pd.MultiIndex.from_arrays(
    [
        col_count_VA1,
        df_VA1.columns.get_level_values('region1'),
        df_VA1.columns.get_level_values('region2'),
        df_VA1.columns.get_level_values('region3'),
        col_sector_VA1,
        df_VA1.columns.get_level_values('sector2')
    ],
    names = rscn2
)

df_VA2 = pd.DataFrame(
    df_VA1.values,
    index=row_VA1,
    columns=col_VA1)

df_VA2

country                                           AT  \
region1                                           R1   
region2                                         AT11   
region3                                   Burgenland   
sector1                                           1    
sector2                                  Agriculture   
sector1 sector2                                        
60      Compensation_of_employees          16.821450   
61      Other_value_added                 148.117717   
62      Taxes_less_subsidies_on_products   12.480693   
65      International_trade_margin          2.475388   

country                                                                      \
region1                                                                       
region2                                                                       
region3                                                                       
sector1                                                                  2    
sector2                                  Mining_quarrying_and_energy_supply   
sector1 sector2                                                               
60      Compensation_of_employees                                 59.619464   
61      Other_value_added                                        153.736256   
62      Taxes_less_subsidies_on_products                          34.996912   
65      International_trade_margin                                10.405807   

country                                                              \
region1                                                               
region2                                                               
region3                                                               
sector1                                                          3    
sector2                                  Food_beverages_and_tobacco   
sector1 sector2                                                       
60      Compensation_of_employees                         58.876294   
61      Other_value_added                                 83.816406   
62      Taxes_less_subsidies_on_products                  17.728262   
65      International_trade_margin                         2.579398   

country                                                            \
region1                                                             
region2                                                             
region3                                                             
sector1                                                        4    
sector2                                  Textiles_and_leather_etc   
sector1 sector2                                                     
60      Compensation_of_employees                       13.744242   
61      Other_value_added                               10.747893   
62      Taxes_less_subsidies_on_products                 4.008108   
65      International_trade_margin                       1.521987   

country                                                                                         \
region1                                                                                          
region2                                                                                          
region3                                                                                          
sector1                                                                                     5    
sector2                                  Coke_refined_petroleum_nuclear_fuel_and_chemicals_etc   
sector1 sector2                                                                                  
60      Compensation_of_employees                                                 54.430339      
61      Other_value_added                                                         81.767108      
62      Taxes_less_subsidies_on_products                                          15.4529

Now we group and sum for rows and columns. First we create a dataframe with summed rows (summing all lines with same country and sector, except for Italy) and then we sum the columns of this dataframe (summing all columns with the same country and sector, except for Italy) and store in another new dataframe.

In [14]:
# First we remove from the dataframe the italian regions and we sum all rows with the same country and sector

df_IO_rows_grouped = df_IO2.loc[df_IO2.index.get_level_values('country') != 'IT', ].groupby(level=['country', 'sector1', 'sector2']).sum()

# When we do this, we end losing the regional index. However the italian regions dataframe will have this information. So we should add this information
# to the dataframe, but as an empty value. That's why our next step should be remaking the multindex:


empty_array = np.full(df_IO_rows_grouped.index.get_level_values('country').shape, np.nan)

# regional-sector's code name with country: rscn2 = ['country','region1', 'region2', 'region3', 'sector1','sector2'] 

row_IO_grouped = pd.MultiIndex.from_arrays(
    [
        df_IO_rows_grouped.index.get_level_values('country'),
        empty_array,
        empty_array,
        empty_array,
        df_IO_rows_grouped.index.get_level_values(1),
        df_IO_rows_grouped.index.get_level_values(2)
    ],
    names = rscn2
)

# And we input this multindex in the intermediary dataframe:

df_IO_rows_grouped = pd.DataFrame(
    df_IO_rows_grouped.values,
    index=row_IO_grouped,
    columns= df_IO_rows_grouped.columns)


# Next, we concatenate the italian-regions in the intermediary dataframe:

df_IO_rows_grouped = pd.concat([df_IO_rows_grouped, df_IO2.loc[df_IO2.index.get_level_values('country') == 'IT', ] ], axis = 0)

# As a result we have a dataframe will rows summed by country and sector, except for Italy!

# Now we do the same thing for the columns.
# For this, we should use the groupby function. However, this function only works vertically. An easy fix for this would be transposing the dataframe, applying the groupby
# function and transposing the dataframe back:

df_IO_grouped = df_IO_rows_grouped.loc[: , df_IO_rows_grouped.columns.get_level_values('country') != 'IT'].T.groupby(level=['country', 'sector1', 'sector2']).sum().T

# We again add a multindex with empty regional labels:

empty_array = np.full(df_IO_grouped.columns.get_level_values('country').shape, np.nan)

col_IO_grouped = pd.MultiIndex.from_arrays(
    [
        df_IO_grouped.columns.get_level_values('country'),
        empty_array,
        empty_array,
        empty_array,
        df_IO_grouped.columns.get_level_values(1),
        df_IO_grouped.columns.get_level_values(2)
    ],
    names = rscn2
)

# Joining the multindex to the dataframe

df_IO_grouped = pd.DataFrame(
    df_IO_grouped.values,
    index=df_IO_grouped.index,
    columns= col_IO_grouped)

# And finally we concantenate the columns with Italian regions back on the dataframe!

df_IO_grouped = pd.concat([df_IO_grouped, df_IO_rows_grouped.loc[: , df_IO_rows_grouped.columns.get_level_values('country') == 'IT'] ], axis = 1)

df_IO_grouped


country                                                                                                AT  \
region1                                                                                               NaN   
region2                                                                                               NaN   
region3                                                                                               NaN   
sector1                                                                                                1    
sector2                                                                                       Agriculture   
country region1 region2 region3  sector1 sector2                                                            
AT      NaN     NaN     NaN      1       Agriculture                                         1.551756e+03   
                                 2       Mining_quarrying_and_energy_supply                  1.278029e+02   
                                 3       Food_beverages_and_tobacco                          3.194744e+02   
                                 4       Textiles_and_leather_etc                            3.645577e+00   
                                 5       Coke_refined_petroleum_nuclear_fuel_and_chemica...  6.430584e+01   
...                                                                                                   ...   
IT      R162    ITG2    Sardegna 11      Hotels_and_restaurant                               4.849663e-08   
                                 12      Transport_storage_and_communication                 1.669259e-02   
                                 13      Financial_intermediation                            3.560630e-03   
                                 14      Real_estate_renting_and_busine_activitie            2.682440e-03   
                                 15      Non-Market_Service                                  1.151186e-02   

country                                                                                                                         \
region1                                                                                                                          
region2                                                                                                                          
region3                                                                                                                          
sector1                                                                                                                     2    
sector2                                                                                     Mining_quarrying_and_energy_supply   
country region1 region2 region3  sector1 sector2                                                                                 
AT      NaN     NaN     NaN      1       Agriculture                                                                 10.134744   
                                 2       Mining_quarrying_and_energy_supply                                       15671.014698   
                                 3       Food_beverages_and_tobacco                                                  13.870516   
                                 4       Textiles_and_leather_etc                                                     2.695573   
                                 5       Coke_refined_petroleum_nuclear_fuel_and_chemica...                          90.364126   
...                                                                                                                        ...   
IT      R162    ITG2    Sardegna 11      Hotels_and_restaurant                                                        0.000001   
                                 12      Transport_storage_and_communication                                          0.193616   
                                 13      Financial_intermediation                      

In [15]:
# First we remove from the dataframe the italian regions and we sum all rows with the same country and sector

df_Y_rows_grouped = df_Y2.loc[df_Y2.index.get_level_values('country') != 'IT', ].groupby(level=['country', 'sector1', 'sector2']).sum()

# When we do this, we end losing the regional index. However the italian regions dataframe will have this information. So we should add this information
# to the dataframe, but as an empty value. That's why our next step should be remaking the multindex:

empty_array = np.full(df_Y_rows_grouped.index.get_level_values('country').shape, np.nan)

row_Y_grouped = pd.MultiIndex.from_arrays(
    [
        df_Y_rows_grouped.index.get_level_values('country'),
        empty_array,
        empty_array,
        empty_array,
        df_Y_rows_grouped.index.get_level_values(1),
        df_Y_rows_grouped.index.get_level_values(2)
    ],
    names = ['country','region1', 'region2', 'region3', 'sector1','sector2']
)

# And we input this multindex in the intermediary dataframe:

df_Y_rows_grouped = pd.DataFrame(
    df_Y_rows_grouped.values,
    index=row_Y_grouped,
    columns= df_Y_rows_grouped.columns)

# Next, we concatenate the italian-regions in the intermediary dataframe:

df_Y_rows_grouped = pd.concat([df_Y_rows_grouped, df_Y2.loc[df_Y2.index.get_level_values('country') == 'IT', ] ], axis = 0)

# As a result we have a dataframe will rows summed by country and sector, except for Italy!

# Now we do the same thing for the columns.
# For this, we should use the groupby function. However, this function only works vertically. An easy fix for this would be transposing the dataframe, applying the groupby
# function and transposing the dataframe back:

df_Y_grouped = df_Y_rows_grouped.loc[: , df_Y_rows_grouped.columns.get_level_values('country') != 'IT'].T.groupby(level=['country', 'sector1', 'sector2']).sum().T

# We again add a multindex with empty regional labels:

empty_array = np.full(df_Y_grouped.columns.get_level_values('country').shape, np.nan)

col_Y_grouped = pd.MultiIndex.from_arrays(
    [
        df_Y_grouped.columns.get_level_values('country'),
        empty_array,
        empty_array,
        empty_array,
        df_Y_grouped.columns.get_level_values(1),
        df_Y_grouped.columns.get_level_values(2)
    ],
    names = ['country','region1', 'region2', 'region3', 'sector1','sector2']
)

# Joining the multindex to the dataframe

df_Y_grouped = pd.DataFrame(
    df_Y_grouped.values,
    index=df_Y_grouped.index,
    columns= col_Y_grouped)

# And finally we concantenate the columns with Italian regions back on the dataframe!

df_Y_grouped = pd.concat([df_Y_grouped, df_Y_rows_grouped.loc[: , df_Y_rows_grouped.columns.get_level_values('country') == 'IT'] ], axis = 1)

df_Y_grouped


country                                                                                                                                                 AT  \
region1                                                                                                                                                NaN   
region2                                                                                                                                                NaN   
region3                                                                                                                                                NaN   
sector1                                                                                                                                                 16   
sector2                                                                                     Final_consumption_expenditure_by_households_and_non-profit_org   
country region1 region2 region3  sector1 sector2                                                                                                             
AT      NaN     NaN     NaN      1       Agriculture                                                                               2031.389317               
                                 2       Mining_quarrying_and_energy_supply                                                        4402.638818               
                                 3       Food_beverages_and_tobacco                                                                6088.439206               
                                 4       Textiles_and_leather_etc                                                                   166.223678               
                                 5       Coke_refined_petroleum_nuclear_fuel_and_chemica...                                        1022.832335               
...                                                                                                                                                    ...   
IT      R162    ITG2    Sardegna 11      Hotels_and_restaurant                                                                        0.000125               
                                 12      Transport_storage_and_communication                                                          2.699747               
                                 13      Financial_intermediation                                                                     0.037178               
                                 14      Real_estate_renting_and_busine_activitie                                                     0.073347               
                                 15      Non-Market_Service                                                                           0.103074               

country                                                                                                                                  \
region1                                                                                                                                   
region2                                                                                                                                   
region3                                                                                                                                   
sector1                                                                                                                              17   
sector2                                                                                     Final_consumption_expenditure_by_government   
country region1 region2 region3  sector1 sector2                                                                                          
AT      NaN     NaN     NaN      1       Agriculture                                                                       8.697427e+01   
                                 2       Mi

In [16]:
df_VA_grouped = df_VA2.loc[: , df_VA2.columns.get_level_values('country') != 'IT'].T.groupby(level=['country', 'sector1', 'sector2']).sum().T

# Again we create a multindex with empty values for regions:

empty_array = np.full(df_VA_grouped.columns.get_level_values('country').shape, np.nan)

col_VA_grouped = pd.MultiIndex.from_arrays(
    [
        df_VA_grouped.columns.get_level_values('country'),
        empty_array,
        empty_array,
        empty_array,
        df_VA_grouped.columns.get_level_values(1),
        df_VA_grouped.columns.get_level_values(2)
    ],
    names = ['country','region1', 'region2', 'region3', 'sector1','sector2']
)

# Joining the multindex to the dataframe:

df_VA_grouped = pd.DataFrame(
    df_VA_grouped.values,
    index=df_VA_grouped.index,
    columns= col_VA_grouped)

# Joining the columns for italian regions to the dataframe:

df_VA_grouped = pd.concat([df_VA_grouped, df_VA2.loc[: , df_VA2.columns.get_level_values('country') == 'IT'] ], axis = 1)

df_VA_grouped

country                                            AT  \
region1                                           NaN   
region2                                           NaN   
region3                                           NaN   
sector1                                            1    
sector2                                   Agriculture   
sector1 sector2                                         
60      Compensation_of_employees          311.748877   
61      Other_value_added                 3251.060802   
62      Taxes_less_subsidies_on_products   269.638708   
65      International_trade_margin          45.070561   

country                                                                      \
region1                                                                       
region2                                                                       
region3                                                                       
sector1                                                                  2    
sector2                                  Mining_quarrying_and_energy_supply   
sector1 sector2                                                               
60      Compensation_of_employees                               2030.683422   
61      Other_value_added                                       5410.231462   
62      Taxes_less_subsidies_on_products                        1221.334141   
65      International_trade_margin                               148.191268   

country                                                              \
region1                                                               
region2                                                               
region3                                                               
sector1                                                          3    
sector2                                  Food_beverages_and_tobacco   
sector1 sector2                                                       
60      Compensation_of_employees                       1962.513707   
61      Other_value_added                               2873.826930   
62      Taxes_less_subsidies_on_products                 602.206151   
65      International_trade_margin                        75.291289   

country                                                            \
region1                                                             
region2                                                             
region3                                                             
sector1                                                        4    
sector2                                  Textiles_and_leather_etc   
sector1 sector2                                                     
60      Compensation_of_employees                      526.563841   
61      Other_value_added                              426.731463   
62      Taxes_less_subsidies_on_products               155.847605   
65      International_trade_margin                      26.016885   

country                                                                                         \
region1                                                                                          
region2                                                                                          
region3                                                                                          
sector1                                                                                     5    
sector2                                  Coke_refined_petroleum_nuclear_fuel_and_chemicals_etc   
sector1 sector2                                                                                  
60      Compensation_of_employees                                               2218.925685      
61      Other_value_added                                                       3376.996928      
62      Taxes_less_subsidies_on_products                                      

We conclude the grouping process. Now we can perform operations on the new matrices:
Again, we first transform the dataframes in Numpy arrays:

In [17]:
IOr = df_IO_grouped.to_numpy()
Yr = df_Y_grouped.to_numpy() 
VAr = df_VA_grouped.to_numpy()

Technical coefficients matrix:

In [18]:
VBPr_T = np.sum(IOr, axis=0) + np.sum(VAr, axis=0)

Ar = IOr/VBPr_T

Identity matrix and Leontieff's inverse matrix:

In [19]:
Ir = np.eye(Ar.shape[0])
Lr = np.linalg.inv(Ir - Ar)

Checking values:

In [21]:
Yr_s = np.sum(Yr, axis = 1) 

In [22]:
error = VBPr_T - Lr @ Yr_s
sum_error = np.sum(error)
sum_error_abs = np.sum(abs(error))
print(sum_error, sum_error_abs)

5.1854538440920805e-09 6.773414717997639e-08


Creating a dataframe for the technical coefficients matrix and the Leontieff's inverse matrix:

In [23]:
df_Ar = pd.DataFrame(
    Ar,
    index=df_IO_grouped.index,
    columns=df_IO_grouped.columns
)

df_Lr = pd.DataFrame(
    Lr,
    index=df_IO_grouped.index,
    columns=df_IO_grouped.columns
)

EXPORT DATA FOR PART 2:

In this section you can export the data for each of the cleaned tables, as well as the new matrices, as a .csv file. Pay attention to the settings used, which separator is being used and decimal, or if either a decimal point or decimal comma is being used. The ideal setting will depend on how your microsoft excel software is configured:

Export IO table:

In [47]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "IOr.csv"

path = f"data/processed/{name}"

df_IO_grouped.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export Added Value Table:

In [48]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "VAr.csv"

path = f"data/processed/{name}"

df_VA_grouped.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export final demand table:

In [49]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "Yr.csv"

path = f"data/processed/{name}"

df_Y_grouped.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export Technical Coefficients table:

In [50]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "Ar.csv"

path = f"data/processed/{name}"

df_Ar.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export Leontieff's table:

In [51]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "Lr.csv"

path = f"data/processed/{name}"

df_Lr.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

    Part 3

In this section we will be doing the hypothetical extraction of the Sicilian economy. In this methodology, we will be nullifying the rows and columns corresponding to the Sicilian sectors in both the technical coefficients matrix (A) and the final demand matrix (y). The economic interpretation is a complete lockdown of the Sicilian economy. Sicily will neither demand intermediate and final products from the other sectors, as it will not provide products to saciate the demand for intermediate and final goods. After that, we compute the Gross Value of Production by Region (GVPR) and Country. But for this numbers to mean anything, we will first need a baseline value for the GVPR. 

Baseline: Gross Value of Production by Region

In [24]:
df_VBPr = pd.DataFrame(
    VBPr_T,
    index = df_IO_grouped.index
                       )

df_VBPr_rtotal = df_VBPr.groupby(level = ['country', 'region1', 'region2', 'region3'], dropna = False).sum()
df_VBPr_rtotal

,,,,0
country,region1,region2,region3,
AT,NaN,NaN,NaN,5.158086e+05
AU,NaN,NaN,NaN,1.411597e+06
BE,NaN,NaN,NaN,6.975247e+05
BG,NaN,NaN,NaN,7.616724e+04
BR,NaN,NaN,NaN,1.965267e+06
...,...,...,...,...
TU,NaN,NaN,NaN,7.988959e+05
TW,NaN,NaN,NaN,5.810755e+05
UK,NaN,NaN,NaN,2.869530e+06


Here, we simply create a list with all Italian regions:

In [57]:
regions_italy = df_VBPr_rtotal.loc['IT',:].index.to_frame(index = False).loc[:,'region3'].to_list()

print(regions_italy)

['Piemonte', 'Valle_dAosta_Vallee_dAoste', 'Liguria', 'Lombardia', 'Provincia_Autonoma_BolzanoBozen', 'Provincia_Autonoma_Trento', 'Veneto', 'FriuliVenezia_Giulia', 'EmiliaRomagna', 'Toscana', 'Umbria', 'Marche', 'Lazio', 'Abruzzo', 'Molise', 'Campania', 'Puglia', 'Basilicata', 'Calabria', 'Sicilia', 'Sardegna']


Zeroing the rows and columns that belong to the Sicilian economy in the techinical coefficients matrix and final demand matrix:

In [60]:
df_Ar1 = df_Ar

print(regions_italy[-2])

row = df_Ar1.index.get_level_values('region3') == regions_italy[-2]
col = df_Ar1.columns.get_level_values('region3') == regions_italy[-2]

df_Ar1.loc[row, :] = 0
df_Ar1.loc[:, col] = 0


Sicilia


In [61]:
df_Yr1 = df_Y_grouped

row = df_Yr1.index.get_level_values('region3') == regions_italy[-2]
col = df_Yr1.columns.get_level_values('region3') == regions_italy[-2]

df_Yr1.loc[row, :] = 0
df_Yr1.loc[:, col] = 0

In [63]:
Ar1 = df_Ar1.to_numpy()
Yr1 = df_Yr1.to_numpy()

Generating the Leontieff Inverse Matrix:

In [64]:
Lr1 = np.linalg.inv(Ir - Ar1)

Summing the final demand matrix so it becomes a vector:

In [65]:
Yr1_s = np.sum(Yr1, axis = 1) 

Generating the Gross Value of Production vector:

In [66]:
VBPr1 = Lr1 @ Yr1_s

For this new vector, we do the same process that we did at the beggining for the original Gross Value of Production vector:

In [67]:
df_VBPr1 = pd.DataFrame(
    VBPr1,
    index = df_IO_grouped.index
                       )

df_VBPr1_rtotal = df_VBPr1.groupby(level = ['country', 'region1', 'region2', 'region3'], dropna = False).sum()
df_VBPr1_rtotal

,,,,0
country,region1,region2,region3,
AT,NaN,NaN,NaN,5.082058e+05
AU,NaN,NaN,NaN,1.411360e+06
BE,NaN,NaN,NaN,6.954929e+05
BG,NaN,NaN,NaN,7.602731e+04
BR,NaN,NaN,NaN,1.964442e+06
...,...,...,...,...
TU,NaN,NaN,NaN,7.971150e+05
TW,NaN,NaN,NaN,5.808245e+05
UK,NaN,NaN,NaN,2.866633e+06


Now we can compute how much each region loss in Gross Value of Production (in percentage):

In [68]:
df_VBPr_pchange = ((df_VBPr1_rtotal - df_VBPr_rtotal)/df_VBPr_rtotal)*100

In [69]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "HE_Sicilia_" + file_year + ".csv" 

path = f"data/processed/{name}"

df_VBPr_pchange.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Part 4

In this section, we will be constructing a closed model, in order to generate economic growth results for the Sicilian economy.

The main difference is that in the open model (the one we developed before), consumption is considered an exogenous variable, while in the closed model, families are one of the factors of production that enter in the technical coefficients matrix. In both models, we apply a final demand vector (exogenous shock) to the Leontieff matrix and check the impact at the Gross Value of Production vector. From that, we can generate an Added Value vector or change in employment vector.

The main practical implication is that in the open model, we cannot estimate the aggregate GDP impact of an exogenous shock to the final demand. We can only generate impacts on gross value of production, added value, change in employment for each regional sector separately. If we sum all the values at the generated added value vector, it will correspond to the sum of all the values at the original exogenous shock vector. This is basically an accounting identity, the increase in national income corresponds to the increase in aggregate value.

However, in the closed model, as we have a mechanism for an increase in expenditure, in which consumers spend more with an increase in their income (endogenous consumption), we can generate the increase (or decrease if we are removing the sector) in the total GDP. This model better captures the circular flow of money in an economy by making consumption endogenous.

To generate results, we will have to make a few additional important assumptions:
1- Workers hired by a regional sector spend in the same region that they were hired: We make this assumption because we do not have information on the place of origin of the workers. We just have the information of which sector hired them. So we will assume they are all from the region that hired them.
2- Profits are not disposable income to the families: Profits are an income to the firms, and not to the families. Profits may be retained for future investment, or may be distributed to foreigners. Assuming that they go directly to the families of the region in question would be an unrealistic assumption. As we do not have information on where do profits go, we should treat as a "leakage" of economic system. The same thing goes for international trade margins.

We start by generating a consumption matrix, by filtering the final demand dataframe:

In [62]:
mask = df_Y_grouped.columns.get_level_values('sector2') == 'Final_consumption_expenditure_by_households_and_non-profit_org'
df_Cr = df_Y_grouped.loc[:, mask]

Our technical coefficients matrix needs to be a square matrix. However, while we have a consumption sector for each region, we have only only one workers sector for all region in the EU. We need to segregate the different workers by different regions. Using assumption one, we can generate this matrix, turning the technical coefficients matrix square:

Generating the workers compensation matrix:

In [63]:
# First, we extract the compensation of employess vector:

mask = df_VA_grouped.index.get_level_values('sector2') == 'Compensation_of_employees'
df_Wr = df_VA_grouped.loc[mask, :]

# We now need to generate an index based on country and region for the rows of our new dataframe:

row_reg_VA =  df_Wr.T.index.to_frame(index=False) # We transpose the indexes in column to obtain the regions in the rows
row_reg_VA  = row_reg_VA.groupby(by = ['country','region1', 'region2','region3'], dropna = False).first() # We group by country and region
row_reg_VA.loc[:,:] = df_VA_grouped.index.to_frame(index=False).iloc[0,:].T.values # We input the name of the workers sector in the index
row_reg_VA = row_reg_VA.reset_index() # We transform the data in a dataframe with no indexes
row_reg_VA = pd.MultiIndex.from_frame(
    row_reg_VA, names = row_reg_VA.columns.tolist()
) # Now we create a multi index for the rows


# Now, we generate a dataframe filled with zeros, with the correct dimensions:

df_Wr_assumption = pd.DataFrame(
    0,
    index= row_reg_VA,
    columns= df_VA_grouped.columns
)

# And we fill the dataframe using the values in the vector in the correponding cells:

df_Wr_assumption = df_Wr_assumption.astype(float)


# regional-sector's code name with country: rscn2 = ['country','region1', 'region2', 'region3', 'sector1','sector2'] 
position_list = rscn2.index('region3')

i = 0
while i < df_Wr_assumption.shape[0]:
    j = 0
    while j < df_Wr_assumption.shape[1]:
        if df_Wr_assumption.index[i][0:position_list] == df_Wr_assumption.columns[j][0:position_list]:
            df_Wr_assumption.iloc[i,j] = df_Wr.iloc[0,j]
        j += 1
    i +=1

# We just need to do a final adjustment: We need to place all the Italy lines in last, because this is the format in which the 
# consumption matrix is:

df_Wr_assumption = pd.concat([df_Wr_assumption.loc[df_Wr_assumption.index.get_level_values('country') != 'IT', ],
                            df_Wr_assumption.loc[df_Wr_assumption.index.get_level_values('country') == 'IT', ]], axis = 0)


We were able to generate a disaggregation of the workers sectors. Now we can perform the necessary calculations for our model.

We start by computing the propensity to consume matrix:

In [64]:
Wr = df_Wr_assumption.to_numpy()
Cr = df_Cr.to_numpy()

m = Wr.sum(axis=1)

prop_Cr = Cr/m


We also compute the technical coefficients of workers:

In [65]:
coef_Wr = Wr/VBPr_T

Now we concatenate the matrices generated to the original technical coefficients matrix.

In [66]:
# However notice the shape of the matrices generated
print(coef_Wr.shape, prop_Cr.shape)
# We need to complete the empty spots in the concatenated matrix with a zeros matrix

# Creating the zeros matrix:

zeros = np.zeros((coef_Wr.shape[0], prop_Cr.shape[1]))

# Concatanating to Ar matrix:

Ac = np.concatenate(( np.concatenate((Ar, prop_Cr), axis=1), np.concatenate((coef_Wr, zeros), axis=1) ), axis = 0)


(61, 854) (854, 61)


Identity matrix and Leontieff's inverse matrix:

In [67]:
Ic = np.eye(Ac.shape[0])
Lc = np.linalg.inv(Ic - Ac)

Checking the results:

In [68]:
# First we shall remove the aggregate consumption from the final demand vector:

f = np.sum(Yr, axis = 1) - np.sum(Cr, axis =1) 

# However, notice that we have extra lines that need to be filled. The Leontieff inverse matrix is bigger in height than the original 
# final demand matrix, due to the lines that correspond to the workers sector. As we do not have data on the final demand for autonomous
# workers' production (the line corresponding to the aggregate value generated from final demand direct to families is empty in the input
# - output table), we assume it to be zero.

zeros_vector = np.zeros((zeros.shape[0], ))

f = np.concatenate((f , zeros_vector), axis = 0)

# For the gross value of production vector we must do something similar. We must concatenate the wages vector

x = np.concatenate((VBPr_T , m), axis = 0)

# Now we can if the results were computed correctly:

error = x  - Lc @ f
sum_error = np.sum(error)
sum_error_abs = np.sum(abs(error))
print(sum_error, sum_error_abs)

-2.4557955669024523e-09 9.099062125983437e-08


Creating a dataframe for the technical coefficients matrix and for the Leontieff matrix:

In [69]:
row_Ac = pd.concat( (df_Ar.index.to_frame(index=False), row_reg_VA.to_frame(index=False)), axis = 0)

row_Ac = pd.MultiIndex.from_frame(
    row_Ac, names = row_Ac.columns.to_list()
)

col_Ac = pd.concat( (df_Ar.columns.to_frame(index=False), df_Cr.columns.to_frame(index=False)), axis = 0)

col_Ac = pd.MultiIndex.from_frame(
    col_Ac, names = col_Ac.columns.to_list()
)

df_Ac = pd.DataFrame(
    Ac,
    index = row_Ac, 
    columns= col_Ac
    )

df_Lc = pd.DataFrame(
    Lc,
    index = row_Ac,
    columns = col_Ac
)

EXPORT DATA FOR PART 4:

In this section you can export the data for each of the cleaned tables, as well as the new matrices, as a .csv file. Pay attention to the settings used, which separator is being used and decimal, or if either a decimal point or decimal comma is being used. The ideal setting will depend on how your microsoft excel software is configured:

Export IO table (this is the same one for Part 2):

In [70]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "IOr.csv"

path = f"data/processed/{name}"

df_IO_grouped.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export Added Value table (this is the same one for Part 2):

In [72]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "VAr.csv"

path = f"data/processed/{name}"

df_VA_grouped.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export final demand table (this is the same one for Part 2):

In [73]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "Yr.csv"

path = f"data/processed/{name}"

df_Y_grouped.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export Technical Coefficients Matrix

In [74]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "Ac.csv"

path = f"data/processed/{name}"

df_Ac.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")

Export Leontieff inverse matrix

In [75]:
# Default settings: , sep = ',', decimal = '.', header = "True", index = "True"

name = "Lc.csv"

path = f"data/processed/{name}"

df_Lc.to_csv(path, sep = ';', decimal = ',', header = "True", index = "True")